# Load exported results (`.nc`, `.h5`, `.npz`)

**Export Results** (Sequence mode) writes the result of the *current*
simulation as plain data files: xarray NetCDF (`.nc`, default, with an
analysis notebook), HDF5 (`.h5`) or NumPy (`.npz`), and optionally
Bruker / DICOM / NIfTI images. These files

* can be read **without Bloch Simulator** (xarray, h5py, NumPy, MATLAB),
* contain the signal, k-space, reconstructed images, final magnetization,
  B1 maps and the simulation parameters as JSON metadata,
* **cannot** be re-opened as a GUI session. They do not contain the phantom
  object, the B1 field files or the GUI controls. Use a Bloch project
  ([load_bloch_project.ipynb](load_bloch_project.ipynb)) for that.

Free Mode exports (HDF5 with `mx`, `my`, `mz` per time, position and
frequency) are covered by [read_hdf5_examples.py](read_hdf5_examples.py).

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

# Change this to your exported file (.nc, .h5 or .npz).
RESULT_FILE = Path("result.nc")

## NetCDF (`.nc`)

NetCDF has no complex data type, so complex variables (signal, k-space,
images) are stored as `<name>_real` and `<name>_imag`. The helper joins them
again. The dimensions are named, e.g. `adc`, `coil`, `pool`, `phase_y`,
`read_x`.

In [ ]:
def load_netcdf(filename):
    """Return the xarray Dataset (complex variables restored) and its metadata."""
    with xr.open_dataset(filename) as stored:
        dataset = stored.load()
    for name in list(dataset.data_vars):
        base = name[:-5]
        if name.endswith("_real") and f"{base}_imag" in dataset:
            dataset[base] = dataset[name] + 1j * dataset[f"{base}_imag"]
            dataset = dataset.drop_vars([name, f"{base}_imag"])
    metadata = json.loads(dataset.attrs.get("metadata_json", "{}"))
    return dataset, metadata


if RESULT_FILE.suffix == ".nc":
    dataset, metadata = load_netcdf(RESULT_FILE)
    print(sorted(dataset.data_vars))

## HDF5 (`.h5`) and NumPy (`.npz`)

Both hold the same arrays as plain datasets and the metadata as one JSON
string `metadata_json` (a file attribute in HDF5, an entry in NPZ).

In [ ]:
def load_arrays(filename):
    """Return a dict of arrays and the metadata from an HDF5 or NPZ export."""
    if Path(filename).suffix == ".npz":
        with np.load(filename, allow_pickle=False) as stored:
            arrays = {key: stored[key] for key in stored.files}
        metadata = json.loads(str(arrays.pop("metadata_json")))
    else:
        import h5py

        with h5py.File(filename, "r") as handle:
            arrays = {key: handle[key][()] for key in handle.keys()}
            metadata = json.loads(handle.attrs["metadata_json"])
    return arrays, metadata


if RESULT_FILE.suffix in (".h5", ".npz"):
    arrays, metadata = load_arrays(RESULT_FILE)
    for name, values in sorted(arrays.items()):
        print(f"{name}: {np.shape(values)} {np.asarray(values).dtype}")

## Simulation parameters

The metadata is the same for all three formats.
`metadata["sequence_definitions"]` holds the sequence parameters (flip angle
in degrees, matrix, FOV in m, TR/TE in s, …). Arrays are stored in their text
form, e.g. `"[64. 64.]"`. Nucleus, B0, the number of simulated spins and, for
a run of a parameter loop, `parameter_loop` are stored next to it.

In [ ]:
definitions = metadata.get("sequence_definitions", {})
print("Sequence:", definitions.get("Name", metadata.get("sequence_source")))
for key in ("FlipAngleDeg", "MatrixSize", "FOV", "TR", "TE"):
    print(f"  {key}: {definitions.get(key)}")
print("Nucleus:", metadata.get("nucleus"), " B0:", metadata.get("field_strength_t"), "T")
print("Simulated spins:", metadata.get("n_simulated_spins"))
print("Parameter loop:", metadata.get("parameter_loop"))
print("All metadata keys:", ", ".join(sorted(metadata)))

## Images, pools and B1 maps (NetCDF)

For Cartesian sequences the export contains `cartesian_kspace` and
`cartesian_image`; with a multi-pool phantom also one image per pool in
`species_cartesian_image`. If a B1 field was used, `tx_sensitivity_map` and
`effective_peak_b1_gauss` are given on the phantom grid.

In [ ]:
if RESULT_FILE.suffix == ".nc":
    print("signal:", dict(dataset["signal"].sizes))
    if "cartesian_image" in dataset:
        image = dataset["cartesian_image"]
        plt.imshow(np.abs(image.values), cmap="gray")
        plt.title(f"Cartesian image {dict(image.sizes)}")
        plt.colorbar()
        plt.show()
    if "species_cartesian_image" in dataset:
        for pool in range(dataset.sizes["pool"]):
            pool_image = dataset["species_cartesian_image"].isel(pool=pool)
            print(f"pool {pool}: max |image| = {float(np.abs(pool_image).max()):.4g}")
    for name in ("tx_sensitivity_map", "effective_peak_b1_gauss"):
        if name in dataset:
            print(f"{name}: {dict(dataset[name].sizes)}")